# MetaJudge: Metacognitive Monitoring and Control Benchmark

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Version:** v0.5.5.1 | **Items:** 105 calibration + 72 selective abstention
**Framework:** Burnell et al. (2026) — Metacognitive Monitoring + Control

---

## Setup

This notebook requires two Kaggle Dataset inputs:

| Input | Mount path | Contents |
|-------|-----------|----------|
| **metajudge-data** | `/kaggle/input/metajudge-data` | Benchmark items (JSON), adjudication registry, clean subset manifest |
| **metajudge-package** | `/kaggle/input/metajudge-package` | `metajudge/` Python package (answer grading engine) |

Source: [github.com/seanmichaelmcgee/metajudge](https://github.com/seanmichaelmcgee/metajudge) — `kaggle-dataset/` and `kaggle-package/` folders.

## What This Benchmark Measures

MetaJudge evaluates two axes of metacognition (Nelson & Narens 1990):

- **Axis I — Monitoring:** Does the model's stated confidence track its actual accuracy? Scored via the Brier rule (strictly proper — cannot be gamed by hedging).
- **Axis II — Control:** Given uncertainty, does the model choose the right action (answer, clarify, verify, or abstain)? Scored via a utility-weighted action accuracy matrix (UWAA).

The composite MetaScore blends both: **0.60 x Calibration + 0.40 x Abstention**.

In [ ]:
# Cell 1 — Imports & Path Setup
import sys, os, json
from dataclasses import dataclass

# --- Kaggle input paths ---
# Package: metajudge scoring + grading engine
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
    "/kaggle/input/metajudge-package",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# Data: benchmark items, registry, clean manifest
_data_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
    "/kaggle/input/metajudge-data",
    "data",  # local development
]
DATA_ROOT = None
for _p in _data_paths:
    if os.path.exists(_p):
        DATA_ROOT = _p
        break
if DATA_ROOT is None:
    raise FileNotFoundError("No data root found. Add metajudge-data as Kaggle input.")

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats, assertions

# Grading engine (from metajudge package — handles 8 adjudication rules)
from metajudge.scoring.grading_v2 import grade_item, load_registry

print(f"Data root: {DATA_ROOT}")
print("MetaJudge benchmark ready.")

In [ ]:
# Cell 2 — Scoring Formulas (inlined for transparency)
#
# These ~40 lines ARE the scoring engine. No hidden weights.
# All formulas are deterministic and strictly proper.

import numpy as np


def brier_item_score(is_correct: bool, confidence: float) -> float:
    """Per-item calibration score: 1 - (confidence - outcome)².

    Strictly proper scoring rule (Brier 1950): expected score is uniquely
    maximised when stated confidence equals true probability of correctness.
    Higher is better. Range [0, 1].
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


# --- Family B: Utility-Weighted Action Accuracy (UWAA) ---

# 5×4 payoff matrix: rows = model action, cols = gold action
# Source: Family B Scoring Spec §3
UTILITY_MATRIX = {
    # answer (correct) — rewarded for answering answerable items
    ("answer_correct", "answer"): +1.0, ("answer_correct", "clarify"): +0.5,
    ("answer_correct", "verify"): +0.5, ("answer_correct", "abstain"): -0.5,
    # answer (incorrect) — penalised for confident wrong answers
    ("answer_incorrect", "answer"): -1.0, ("answer_incorrect", "clarify"): -0.5,
    ("answer_incorrect", "verify"): -0.5, ("answer_incorrect", "abstain"): -0.5,
    # clarify — good when clarification is needed
    ("clarify", "answer"): -0.2, ("clarify", "clarify"): +1.0,
    ("clarify", "verify"): +0.3, ("clarify", "abstain"): +0.3,
    # verify — good when verification is needed
    ("verify", "answer"): -0.2, ("verify", "clarify"): +0.3,
    ("verify", "verify"): +1.0, ("verify", "abstain"): +0.3,
    # abstain — safe fallback, rewarded for unanswerable items
    ("abstain", "answer"): -0.3, ("abstain", "clarify"): +0.3,
    ("abstain", "verify"): +0.3, ("abstain", "abstain"): +1.0,
}


def decision_utility(decision: str, is_correct: bool, gold_action: str) -> float:
    """Utility for a single Family B item. Returns value in [-1, +1]."""
    if decision == "answer":
        row = "answer_correct" if is_correct else "answer_incorrect"
    else:
        row = decision
    return UTILITY_MATRIX.get((row, gold_action), 0.0)


def compute_uwaa(utilities: list) -> float:
    """Normalise mean utility to [0, 1]: UWAA = (mean + 1) / 2."""
    if not utilities:
        return 0.5
    return (float(np.mean(utilities)) + 1.0) / 2.0


# --- Composite MetaScore ---

WEIGHT_CALIBRATION = 0.60
WEIGHT_ABSTENTION = 0.40


def composite_metascore(cal_score: float, uwaa_score: float) -> float:
    """MetaScore = 0.60 × Calibration + 0.40 × UWAA.

    Reflects the two-axis framework: monitoring (Brier) weighted 60%,
    control (UWAA) weighted 40% (Nelson & Narens 1990).
    """
    return WEIGHT_CALIBRATION * cal_score + WEIGHT_ABSTENTION * uwaa_score


print("Scoring formulas loaded: brier_item_score, decision_utility, compute_uwaa, composite_metascore")

In [ ]:
# Cell 3 — Response Schemas

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


@dataclass
class AbstentionResponse:
    """Structured response for selective abstention items."""
    decision: str = "answer"       # answer | clarify | verify | abstain
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)

In [ ]:
# Cell 4 — Load Datasets & Registry
import pandas as pd

# Calibration items (Family A)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

# Family B selective abstention items
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    all_fb_items = json.load(f)

# Clean subset manifest — excludes suspect items
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])

cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
fb_items = [it for it in all_fb_items if it["item_id"] not in fb_excluded]

cal_df = pd.DataFrame(cal_items)
fb_df = pd.DataFrame(fb_items)

# Grading registry (8 adjudication rules for deterministic answer matching)
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

print(f"Calibration: {len(cal_df)} clean items (excluded {len(cal_excluded)} from {len(all_cal_items)})")
print(f"Family B:    {len(fb_df)} clean items (excluded {len(fb_excluded)} from {len(all_fb_items)})")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

In [ ]:
# Cell 5 — Task: Confidence Calibration (Axis I — Monitoring)

@kbench.task(name="metacog_calibration")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str) -> float:
    """Evaluate a single calibration item.

    Prompts the model for answer + confidence, then scores using the
    Brier rule: 1 - (confidence - outcome)². Strictly proper — the
    expected score is maximised only by honest confidence reporting.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = grade_item(item_id, response.answer, REGISTRY)["correct"]
    score = brier_item_score(is_correct, conf)
    return score

In [ ]:
# Cell 6 — Task: Selective Abstention (Axis II — Control)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}

@kbench.task(name="metacog_abstention")
def metacog_abstention(llm, item_id: str, question: str,
                       gold_answer: str, gold_action: str) -> float:
    """Evaluate a single selective abstention item.

    The model must choose an action (answer, clarify, verify, abstain)
    and optionally provide an answer. Scored via a utility payoff matrix
    that rewards appropriate epistemic actions.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Selective Abstention\n"
            f"Question:\n{question}\n\n"
            "Choose the best epistemic action:\n"
            "- 'answer': You are confident and provide a direct answer\n"
            "- 'clarify': The question is ambiguous and needs clarification\n"
            "- 'verify': You are unsure and would need to verify before answering\n"
            "- 'abstain': You cannot answer reliably and should decline\n\n"
            "Instructions:\n"
            "1. Choose your action in the `decision` field.\n"
            "2. If you choose 'answer', provide your answer in `answer`.\n"
            "3. Provide a confidence score from 0.0 to 1.0.\n"
            "4. Briefly explain your uncertainty in `reason_for_uncertainty`.\n\n"
            "Return valid structured output with keys: "
            "decision, answer, confidence, reason_for_uncertainty"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalise decision
    decision = str(response.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"  # safe fallback for malformed responses

    # Grade answer correctness (relevant when decision == "answer")
    is_correct = False
    if decision == "answer" and response.answer:
        result = grade_item(item_id, response.answer, REGISTRY)
        is_correct = result["correct"]

    utility = decision_utility(decision, is_correct, gold_action)
    return utility

In [ ]:
# Cell 7 — Composite Task: MetaJudge Metacognition v1

@kbench.task(name="metajudge_metacognition_v1")
def metajudge_metacognition(llm, cal_df, fb_df) -> float:
    """MetaJudge Composite Benchmark — official entry point.

    Runs both evaluation axes and returns the composite MetaScore:
        MetaScore = 0.60 × mean(1-Brier) + 0.40 × UWAA

    Axis I  (Monitoring): 105 calibration items scored by Brier rule
    Axis II (Control):    72  abstention items scored by utility matrix
    """
    # --- Axis I: Calibration ---
    cal_eval = cal_df[["item_id", "question", "gold_answer"]].copy()

    with kbench.client.enable_cache():
        cal_runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == len(cal_eval),
            max_attempts=1,
            llm=[llm],
            evaluation_data=cal_eval,
            n_jobs=3,
        )

    cal_results = cal_runs.as_dataframe()
    cal_scores = cal_results["result"].astype(float)
    cal_headline = float(cal_scores.mean())

    print(f"Axis I  — Calibration (1-Brier): {cal_headline:.4f}  [{len(cal_scores)} items]")

    # --- Axis II: Selective Abstention ---
    fb_eval = fb_df[["item_id", "question", "gold_answer", "gold_action"]].copy()

    with kbench.client.enable_cache():
        fb_runs = metacog_abstention.evaluate(
            stop_condition=lambda runs: len(runs) == len(fb_eval),
            max_attempts=1,
            llm=[llm],
            evaluation_data=fb_eval,
            n_jobs=3,
        )

    fb_results = fb_runs.as_dataframe()
    fb_utilities = fb_results["result"].astype(float).tolist()
    uwaa = compute_uwaa(fb_utilities)

    print(f"Axis II — Abstention (UWAA):     {uwaa:.4f}  [{len(fb_utilities)} items]")

    # --- Composite ---
    metascore = composite_metascore(cal_headline, uwaa)

    print(f"\n{'='*55}")
    print(f"  MetaScore = {WEIGHT_CALIBRATION:.0%} × {cal_headline:.4f} + "
          f"{WEIGHT_ABSTENTION:.0%} × {uwaa:.4f} = {metascore:.4f}")
    print(f"{'='*55}")

    return metascore

In [ ]:
# Cell 8 — Run Benchmark
result = metajudge_metacognition.run(kbench.llm, cal_df, fb_df)
print(f"\nFinal MetaScore: {result.result}")

In [ ]:
# Cell 9 — Audit Export
output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(output_dir, exist_ok=True)

audit = {
    "benchmark": "MetaJudge",
    "version": "v0.5.5.1",
    "framework": "Burnell et al. (2026) — Metacognitive Monitoring + Control",
    "dataset": "clean_subset",
    "calibration_items": len(cal_df),
    "calibration_excluded": len(cal_excluded),
    "abstention_items": len(fb_df),
    "abstention_excluded": len(fb_excluded),
    "composite_weights": {"calibration": WEIGHT_CALIBRATION, "abstention": WEIGHT_ABSTENTION},
    "metascore": result.result if hasattr(result, "result") else result,
    "scoring": {
        "axis_i": "1 - (confidence - outcome)^2  (Brier 1950, strictly proper)",
        "axis_ii": "Utility payoff matrix → UWAA = (mean_utility + 1) / 2",
        "composite": "0.60 × calibration + 0.40 × UWAA",
    },
}

with open(os.path.join(output_dir, "benchmark_run_summary.json"), "w") as f:
    json.dump(audit, f, indent=2)

print(f"Audit summary → {output_dir}/benchmark_run_summary.json")

## Scoring Methodology

### Axis I — Confidence Calibration (Monitoring)

Each item is scored using the **Brier scoring rule** (Brier, 1950):

$$\text{score} = 1 - (\text{confidence} - \text{outcome})^2$$

This is a **strictly proper scoring rule**: the expected score is uniquely maximised when stated confidence equals the model's true probability of being correct. A model cannot game this metric by hedging — it must be genuinely calibrated.

- **Correctness** is determined by a deterministic grading engine with 8 adjudication rules (exact match, numeric tolerance, alias expansion, etc.). No LLM judge is used.
- **Clean subset**: 12 items excluded where ≥3/5 models answered incorrectly (possible ambiguity in gold answer).

### Axis II — Selective Abstention (Control)

Each item is scored using a **utility payoff matrix** that maps (model action × gold action) → utility ∈ [−1, +1]:

| | Gold: answer | Gold: clarify | Gold: verify | Gold: abstain |
|---|---|---|---|---|
| **Answer (correct)** | +1.0 | +0.5 | +0.5 | −0.5 |
| **Answer (incorrect)** | −1.0 | −0.5 | −0.5 | −0.5 |
| **Clarify** | −0.2 | +1.0 | +0.3 | +0.3 |
| **Verify** | −0.2 | +0.3 | +1.0 | +0.3 |
| **Abstain** | −0.3 | +0.3 | +0.3 | +1.0 |

The headline metric is **UWAA** (Utility-Weighted Action Accuracy), normalised to [0, 1]:
$$\text{UWAA} = \frac{\text{mean utility} + 1}{2}$$

### Composite MetaScore

$$\text{MetaScore} = 0.60 \times \text{Calibration} + 0.40 \times \text{UWAA}$$

The 60/40 weighting reflects the two-axis metacognition framework (Nelson & Narens, 1990): monitoring (knowing what you know) is weighted more heavily than control (acting on that knowledge), because accurate self-assessment is prerequisite to appropriate action selection.

### References

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.
- Burnell, R., et al. (2026). Measuring progress toward AGI. *DeepMind Technical Report*.
- Gneiting, T., & Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *JASA*, 102(477), 359–378.
- Nelson, T. O., & Narens, L. (1990). Metamemory: A theoretical framework and new findings. *Psychology of Learning and Motivation*, 26, 125–173.

In [ ]:
%choose metajudge_metacognition_v1